# 08 — Run the best possession pipeline

This is the organiser-facing executable notebook. It runs the canonical `run_interpolated_flight_experiment.py` pipeline and writes the best observation-only candidate: visible-ball low-motion dead-ball evidence plus interval-level `NO_SWITCH` retention. Evaluation and disagreement analysis live in `08_model_evaluation.ipynb`.

The runtime order is deliberately different from the explanatory notebook numbering:

`raw tracking → optional raw George observation stage → source-aware ball/motion features → scenarios → anchors → transition hypotheses → interval resolver → H/A/D/X`

Notebook 04c explains and audits the George stage. It is numbered as a chapter after 04b, but the runner may execute it before our feature layer because it creates an observation dataframe from the raw camera/tracking inputs. The stage is not a possession labeller: it estimates missing-ball geometry and provenance (`source`, `sigma_m`, `p_video`, `p_carrier`, `p_dead`). Our downstream model uses those values as bounded evidence and uncertainty, not as ground truth.

The pipeline is observation-first: engineered observations → scenario evidence → deterministic anchors → bounded transition hypotheses → final H/A/D/X labels. Kalman estimates missing-ball geometry only; it does not decide possession. Viterbi/HMM/HSMM remain optional decoder diagnostics, not the primary truth-maker. The George carrier/dead-ball HMM is a gap-reconstruction component and is not the same as applying Viterbi to final possession labels.

The raw George stage is opt-in in this notebook. With `RUN_GEORGE_FROM_RAW=1`, the runner calls the vendored `ballfill` functions, creates an in-memory observation table, and passes it into our feature/scenario/transition pipeline. Any generated `BAR-ATH_ball.csv` is an audit artifact, not a required input or a copied label file. With the flag off, this notebook runs the promoted reference model without George evidence. The two modes can therefore be compared without changing the downstream resolver.

The runner also emits transition-signal ablations for notebook 09: post-contact/team-level true-switch confirmation, restart activation, their composition, a broad restart-phase interval resolver, and local restart-boundary guards. The resolver is the structural path we are testing: restart observations → restart phase/evidence → transition constraints → shared interval resolver → final H/A/D/X label. These candidates use observations only and are intentionally diagnostic; the best output is not replaced unless the common evaluator shows a real improvement in macro-F1, spell-F1, and transition F1@25.

The full George construction is explained in [04c_george_ballfill_pipeline.ipynb](04c_george_ballfill_pipeline.ipynb).

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUNNER = ROOT / 'possession_lab' / 'run_interpolated_flight_experiment.py'
RUNNER_MODULE = 'possession_lab.run_interpolated_flight_experiment'
EVALUATOR = ROOT / 'evaluation' / 'run_evaluation.py'
RESULTS = ROOT / 'evaluation' / 'results'
REPORTS = ROOT / 'evaluation' / 'reports'
BEST_NAME = 'Forward terminal-flight dead-ball + NO_SWITCH interval resolver'
BEST_OUTPUT = RESULTS / 'diagnostic_forward_settling_dead_ball.csv'
configured_data_root = os.environ.get('MTAH_DATA_DIR')
data_candidates = [Path(configured_data_root)] if configured_data_root else []
data_candidates += [ROOT / 'data', ROOT.parent / 'barca-hack-repo-staging' / 'data']
DATA_ROOT = next((path for path in data_candidates if (path / 'tactical' / 'BAR-ATH_tactical_atd.txt').exists()), None)
RUN_GEORGE_FROM_RAW = os.environ.get('RUN_GEORGE_FROM_RAW', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
GEORGE_DATA_ROOT = Path(os.environ.get('GEORGE_DATA_DIR', str(DATA_ROOT))) if DATA_ROOT is not None else None
GEORGE_BALLFILL = os.environ.get('GEORGE_BALLFILL')
GEORGE_EVIDENCE_ONLY = os.environ.get('GEORGE_EVIDENCE_ONLY', '1').strip().lower() in {'1', 'true', 'yes', 'on'}
GEORGE_MINIMUM_CONFIDENCE = os.environ.get('GEORGE_MINIMUM_CONFIDENCE')
assert RUNNER.exists(), RUNNER
assert EVALUATOR.exists(), EVALUATOR
assert DATA_ROOT is not None, f'tactical data not found in: {data_candidates}'
print('project root:', ROOT)
print('runner:', RUNNER)
print('data root:', DATA_ROOT)
print('candidate:', BEST_NAME)
print('George raw observation stage:', 'enabled' if RUN_GEORGE_FROM_RAW else 'disabled (reference pipeline)')
print('George data root:', GEORGE_DATA_ROOT if RUN_GEORGE_FROM_RAW else 'not applicable')
print('George explicit table ablation:', GEORGE_BALLFILL or 'disabled')
print('George evidence-only:', GEORGE_EVIDENCE_ONLY if (RUN_GEORGE_FROM_RAW or GEORGE_BALLFILL) else 'not applicable')
print('George minimum confidence:', GEORGE_MINIMUM_CONFIDENCE or 'none')

project root: /Users/luke/Desktop/barca-hack-repo-staging
runner: /Users/luke/Desktop/barca-hack-repo-staging/possession_lab/run_interpolated_flight_experiment.py
data root: /Users/luke/Desktop/barca-hack-repo-staging/data
candidate: Forward terminal-flight dead-ball + NO_SWITCH interval resolver
George ballfill: disabled (reference pipeline)
George evidence-only: not applicable
George minimum confidence: none


## Execute the candidate pipeline

This cell regenerates the frame-aligned candidate and its diagnostic evidence files. No ground-truth labels are read by the runner.

In [2]:
runner_environment = os.environ.copy()
runner_environment['MTAH_DATA_DIR'] = str(DATA_ROOT)
runner_command = [sys.executable, '-m', RUNNER_MODULE]
if RUN_GEORGE_FROM_RAW:
    runner_command += ['--run-george-from-raw', '--george-data-dir', str(GEORGE_DATA_ROOT)]
elif GEORGE_BALLFILL:
    runner_command += ['--george-ballfill', GEORGE_BALLFILL]
    if GEORGE_EVIDENCE_ONLY:
        runner_command += ['--george-evidence-only']
    if GEORGE_MINIMUM_CONFIDENCE:
        runner_command += ['--george-minimum-confidence', GEORGE_MINIMUM_CONFIDENCE]
pipeline = subprocess.run(
    runner_command,
    cwd=ROOT,
    env=runner_environment,
    capture_output=True,
    text=True,
    check=False,
)
if pipeline.stdout:
    print(pipeline.stdout)
if pipeline.stderr:
    print('--- runner stderr ---')
    print(pipeline.stderr)
if pipeline.returncode != 0:
    raise RuntimeError(f'pipeline failed with exit code {pipeline.returncode}; see runner stderr above')
print('pipeline completed')
assert BEST_OUTPUT.exists(), BEST_OUTPUT
print('candidate written:', BEST_OUTPUT)

/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_scenario_bidirectional_interpolated_flight_ensemble.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_missing_ball_trajectory.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_kalman_missing_ball_trajectory.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_coordinate_path_bridge.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_player_dribble_gap_bridge.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_restart_type_specific_interpolated_flight.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_dead_ball_boundary_low_motion.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/diagnostic_visible_ball_low_motion_dead_ball.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/reports/transition_bootstrap_stability

## Outputs to present

- Candidate submission: `evaluation/results/diagnostic_forward_settling_dead_ball.csv`
- Scenario/trust evidence: `evaluation/reports/scenario_confidence_scores.csv`
- Transition evidence: `evaluation/reports/transition_hypothesis_ledger.csv`
- Transition-signal diagnostics: the runner also writes direct, hysteretic-handoff, restart-state, broad restart-phase, and local-boundary guard candidates under `evaluation/results/diagnostic_no_switch_*.csv`; they are ranked in notebook 09 and are not automatically promoted.
- Structural full-match candidate: `evaluation/results/diagnostic_no_switch_restart_phase_interval.csv` applies restart evidence as a constraint layer without allowing restart context to invent possession.
- Raw George observation stage: set `RUN_GEORGE_FROM_RAW=1` (and optionally `GEORGE_DATA_DIR=/path/to/data`) before running this notebook. The runner then creates the ball observation dataframe from raw inputs and passes it downstream; no pre-generated `BAR-ATH_ball.csv` is required.
- Optional explicit-table ablation: run `python -m possession_lab.run_george_ballfill_experiment --ballfill /path/to/BAR-ATH_ball.csv --evidence-only --compare-baseline` only when auditing an already generated table; coordinate filling remains a separate diagnostic experiment, not the promoted candidate.

Run `08_model_evaluation.ipynb` next for public scoring and disagreement plots.

## Organiser-facing run summary

This notebook runs the full-match, observation-first candidate. George's ball observations enter before derived motion features; in evidence-only mode their source, uncertainty, flight, speed, distance, and dead-ball signals reach the missing-ball/scenario/transition layers without replacing tactical coordinates. The runnable order is raw tracking → George/source-aware ball observations → motion/player feature engineering → preliminary observation scenarios → anchor selection → final scenario context → transition hypotheses (H switch, A switch, NO_SWITCH, CONTESTED) → shared interval resolver → optional sequence smoothing → H/A/D/X. Kinson's fitted supervised probabilities are not part of this runner.

The current public winner is diagnostic_forward_settling_dead_ball: Macro-F1 0.8947, Spell-F1 0.6400, Transition F1@25 0.4865. It starts from the visible-ball + NO_SWITCH resolver and adds a bounded forward terminal-flight/scene-settling D-onset signal: speed falls, no plausible receiver arrives, and nearby player activity settles. Direction change is supporting evidence, never a standalone D trigger. This score is computed only by the separate evaluation notebook against the supplied five-minute labels. The runner itself uses no ground truth and writes the same frame-aligned submission format needed for the private full match.